# 从操作数据到随机森林头文件：逐步理解 train_model.py

这个 Notebook 用来解释项目的新训练流程。正式训练入口仍是同目录的 <code>train_model.py</code>。

**从上到下运行所有单元格。** 本机已安装依赖，在 VS Code 中选择 <code>/usr/bin/python3</code> 为内核。本机补装全部依赖的命令：

~~~bash
python -m pip install --user -r software/MLcode/requirements-notebook.txt
~~~

默认使用临时目录中的**模拟数据**；结果用于学习，不能代表小车效果。设置 <code>USE_DEMO_DATA = False</code> 后读取真实 dataset。两种模式的演示输出都保存在临时目录，不覆盖正式模型或固件文件。

<table>
<tr><th>输入</th><th>处理</th><th>输出</th></tr>
<tr><td>多个操作记录文件</td><td>逐行清理、保留文件来源</td><td>完整距离矩阵 X、指令 y</td></tr>
<tr><td>训练部分</td><td>排除常量列，按 ANOVA F 分数选择 K 列</td><td>原始特征索引、K 维输入</td></tr>
<tr><td>K 维训练输入</td><td>随机森林训练、独立验证</td><td>准确率、混淆矩阵、类别概率</td></tr>
<tr><td>冻结特征后的全部有效数据</td><td>重训最终森林、导出并核对 C++</td><td>randomForest.h 和类别到指令映射</td></tr>
</table>

## 0. 导入正式脚本，集中设置学习参数

这里导入模块不会运行 <code>main()</code>，也不会同步固件。清理和森林参数直接读取 Python 文件顶部的大写常量。

本 Notebook 只额外配置演示模式、K 和是否执行 C++ 核对。修改正式脚本后，重启内核并重新运行全部单元格。

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import Counter
import csv
import hashlib
import html
import importlib.util
import tempfile

import joblib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import export_text

# 从项目根目录或 MLcode 目录打开 Notebook 都可以找到脚本。
search_roots = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (p for p in search_roots if (p / "software/MLcode/train_model.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请在 dissertation 项目内打开 Notebook。")

SCRIPT_PATH = PROJECT_ROOT / "software/MLcode/train_model.py"
spec = importlib.util.spec_from_file_location("walkthrough_pipeline", SCRIPT_PATH)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

# ======================== Notebook 配置区 ========================
USE_DEMO_DATA = True                     # False = 读取真正的 dataset
NOTEBOOK_K = pipeline.K                   # 也可改为 12
REAL_INPUT_DIR = pipeline.resolve_path(pipeline.INPUT_DIR)
RUN_CPP_CHECK = pipeline.VERIFY_CPP_EXPORT # 默认调用 g++ 核对预测与概率
# ================================================================

workspace = tempfile.TemporaryDirectory(prefix="dissertation-notebook-")
WORKSPACE = Path(workspace.name)
NOTEBOOK_OUTPUT_DIR = WORKSPACE / "outputs"
NOTEBOOK_OUTPUT_DIR.mkdir()

# 表格只是显示工具，不参与训练或清理。
def show_table(headers, rows):
    head = "".join(f"<th>{html.escape(str(v))}</th>" for v in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def file_digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else None

protected_models = (
    pipeline.resolve_path(pipeline.OUTPUT_DIR) / pipeline.HEADER_FILENAME,
    pipeline.resolve_path(pipeline.FIRMWARE_HEADER),
)
original_model_hashes = {p: file_digest(p) for p in protected_models}

show_table(
    ["配置", "当前值"],
    [
        ["脚本路径", SCRIPT_PATH],
        ["模拟数据模式", USE_DEMO_DATA],
        ["K", NOTEBOOK_K],
        ["距离列数", pipeline.LIDAR_POINT_COUNT],
        ["有效距离", f"0 <= distance < {pipeline.MAX_DISTANCE_MM} mm"],
        ["允许标签", pipeline.TRAIN_LABELS],
        ["忽略标签", pipeline.CLEAN_DROP_LABELS],
        ["标签映射", pipeline.LABEL_REMAP],
        ["森林", f"{pipeline.N_ESTIMATORS} trees, max_depth={pipeline.MAX_DEPTH}"],
        ["Notebook 输出", NOTEBOOK_OUTPUT_DIR],
    ],
)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

## 1. 确认采集文件格式，准备输入

采集程序每行写 <code>d0,d1,...,d199,command</code>，默认共 201 列，无表头。

- 前 200 列是雷达距离，单位毫米；末列是蓝牙指令。
- <code>255</code> 是录制标志，默认清理掉；<code>200</code> 会关闭录制，通常不会写入。
- <code>201</code> 是直行。默认训练标签为 0–15；需要保留或合并直行标签时，在脚本顶部修改 TRAIN_LABELS 或 LABEL_REMAP。
- 只更新索引 0–49 和 151–199。其余列的零正常存在，不代表这一行损坏。
- 距离数组保留没更新的历史值，一行不保证是全部角度新鲜的一整圈数据。

模拟数据使用不连续的标签 <code>{3, 8, 10}</code>，用于展示为什么固件需要类别映射。我们有意加入错误行，让后面能看到清理原因。

In [ ]:
if USE_DEMO_DATA:
    INPUT_DIR = WORKSPACE / "demo_dataset"
    INPUT_DIR.mkdir()
    rng = np.random.default_rng(pipeline.RANDOM_SEED)
    active = np.asarray(pipeline.active_indices(), dtype=int)
    start = len(active) // 5
    signal_indices = active[start:start + min(12, len(active) - start)]
    demo_labels = (3, 8, 10)
    centers = {3: 950, 8: 1800, 10: 2350}
    suffix = pipeline.INPUT_EXTENSIONS[0].upper()

    for recording in range(3):
        folder = INPUT_DIR / f"recording_{recording + 1:02d}"
        folder.mkdir()
        path = folder / f"DATA{suffix}"
        clean_rows = []
        for i in range(120):
            command = demo_labels[i % len(demo_labels)]
            distances = np.zeros(pipeline.LIDAR_POINT_COUNT, dtype=int)
            distances[active] = rng.integers(400, 2800, len(active))
            # 人工让一组角度的距离与标签有统计关系，并加入重叠和噪声。
            signal = centers[command] + rng.normal(0, 370)
            distances[signal_indices] = np.clip(
                signal + rng.integers(-70, 71, len(signal_indices)),
                1, pipeline.MAX_DISTANCE_MM - 1,
            ).astype(int)
            clean_rows.append([*distances.tolist(), command])

        example = clean_rows[0]
        with path.open("w", encoding=pipeline.INPUT_ENCODING, newline="") as handle:
            writer = csv.writer(handle, delimiter=pipeline.INPUT_DELIMITER)
            if pipeline.INPUT_HAS_HEADER:
                writer.writerow([*(f"d{i}" for i in range(pipeline.LIDAR_POINT_COUNT)), "command"])
            writer.writerows(clean_rows)
            writer.writerow(example[:-2])                  # 错误列数
            writer.writerow([*example[:-1], 255])           # 录制标志
            writer.writerow([*example[:-1], 201])           # 默认不纳入训练
            writer.writerow(["NaN", *example[1:]])          # 非有限值
            writer.writerow([-1, *example[1:]])             # 距离越界
            writer.writerow([0] * pipeline.LIDAR_POINT_COUNT + [3]) # 全零扫描
            writer.writerow([1.5, *example[1:]])            # 非整数距离
            missing = example.copy()
            for index in signal_indices:
                missing[index] = 0                         # 部分零，清理后再按选中特征过滤
            writer.writerow(missing)
    print("正在使用模拟数据，不能用于小车部署：", INPUT_DIR)
else:
    INPUT_DIR = REAL_INPUT_DIR
    print("正在使用真实操作数据：", INPUT_DIR)

args = SimpleNamespace(
    k=NOTEBOOK_K, input_dir=INPUT_DIR, output_dir=NOTEBOOK_OUTPUT_DIR,
    firmware_header=pipeline.resolve_path(pipeline.FIRMWARE_HEADER), sync=False,
)
pipeline.validate_config(args)
extensions = {s.lower() for s in pipeline.INPUT_EXTENSIONS}
input_files = sorted(
    p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions
)
if not input_files:
    raise FileNotFoundError("没有输入数据；先放入真实数据，或开启 USE_DEMO_DATA。")
show_table(["文件", "字节数"], [(p.relative_to(INPUT_DIR), p.stat().st_size) for p in input_files])

In [ ]:
# 只预览文本，不把缺少列或含非法值的行直接交给训练。
with input_files[0].open(encoding=pipeline.INPUT_ENCODING, errors="replace") as handle:
    preview = []
    for line_number, line in enumerate(handle, 1):
        if pipeline.INPUT_HAS_HEADER and line_number == 1:
            continue
        if line.strip():
            parts = line.strip().split(pipeline.INPUT_DELIMITER)
            preview.append((line_number, len(parts), parts[:6], parts[-1]))
        if len(preview) == 3:
            break
show_table(["原始行号", "列数", "前 6 个字段", "末列标签"], preview)
print("期待列数 =", pipeline.LIDAR_POINT_COUNT + 1)

## 2. 逐行清理：先保证数据能训练

调用正式脚本的 <code>load_and_clean()</code>。它依次检查列数、数字和有限值、整数标签、允许标签、距离范围、整数距离、全零扫描及零值比例。可选去重仅在文件内执行。

返回值 <code>rows</code> 是完整的距离列加标签；<code>groups</code> 记录每一行来自哪个文件，用于后面按文件划分验证集。<code>clean_report</code> 记录保留和删除原因。

In [ ]:
rows, groups, clean_report = pipeline.load_and_clean(INPUT_DIR)
if len(rows) < pipeline.MIN_CLEAN_ROWS:
    raise ValueError(f"有效行数少于 MIN_CLEAN_ROWS={pipeline.MIN_CLEAN_ROWS}。")

show_table(
    ["文件", "原始行数", "保留行数", "原始标签分布", "列数分布"],
    [
        [
            f["file"], f["counts"]["total"], f["counts"].get("kept", 0),
            f["label_counts_before_filter"], f["observed_column_counts"],
        ]
        for f in clean_report["files"]
    ],
)
reason_names = {
    "wrong_column_count": "列数错误",
    "non_numeric": "含非数字",
    "non_finite": "NaN / inf",
    "non_integer_label": "标签不是整数",
    "dropped_label": "录制等忽略标签",
    "label_not_allowed": "标签不在训练范围",
    "distance_out_of_range": "距离越界",
    "non_integer_distance": "距离不是整数",
    "all_zero_scan": "采集扇区全零",
    "too_many_zeros": "零值比例超过配置",
    "duplicate": "文件内重复",
    "blank": "空行",
    "header": "跳过表头",
}
totals = Counter()
for file_report in clean_report["files"]:
    totals.update(file_report["counts"])
discarded = {k: v for k, v in totals.items() if k not in ("total", "kept") and v}
show_table(["删除 / 跳过原因", "行数"], [(reason_names.get(k, k), v) for k, v in discarded.items()])
if discarded:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.barh(list(discarded), list(discarded.values()), color="#6275c9")
    ax.set_xlabel("Rows skipped")
    ax.set_title("Cleaning reasons")
    fig.tight_layout()
    plt.show()

## 3. 分出 X 和 y，观察一行扫描

如果清理后有 M 行，<code>X.shape = (M, 200)</code>，<code>y.shape = (M,)</code>。
X 是输入距离，y 是要学习的操作指令；文件名不参与产生标签。

灰色区域是当前采集程序未更新的角度列。零值列保留在原始矩阵里，让原始列索引始终对应固件的 distances 数组。

In [ ]:
data = np.asarray(rows, dtype=np.float64)
X = data[:, :-1]
y = data[:, -1].astype(int)
assert X.shape[1] == pipeline.LIDAR_POINT_COUNT
print("清理矩阵 data:", data.shape, "；特征 X:", X.shape, "；标签 y:", y.shape)
labels, counts = np.unique(y, return_counts=True)
show_table(["真实指令", "保留样本数"], zip(labels.tolist(), counts.tolist()))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(np.arange(X.shape[1]), X[0], color="#5265b0", linewidth=1)
if pipeline.USE_FRONT_SECTOR_ONLY:
    axes[0].axvspan(
        pipeline.LIDAR_POINT_COUNT // 4,
        3 * pipeline.LIDAR_POINT_COUNT // 4,
        color="#d9dde5", alpha=0.6, label="Not collected",
    )
axes[0].set(xlabel="Original distance-column index", ylabel="Distance (mm)",
            title=f"One cleaned sample; command={y[0]}")
if pipeline.USE_FRONT_SECTOR_ONLY:
    axes[0].legend()
axes[1].bar(labels.astype(str), counts, color="#527ca3")
axes[1].set(xlabel="Command", ylabel="Rows", title="Class distribution")
fig.tight_layout()
plt.show()

## 4. 先划分训练和验证，再选特征

正式脚本的 <code>split_rows()</code> 使用文件分组：同一个文件的行不会同时进入训练和验证。
只有一个文件时，按行顺序分成前后两段，并通过 SPLIT_GAP_ROWS 留出间隔。

必须先划分。选特征时如果看到了验证标签，验证成绩就不能独立评价选择过程。相邻帧和复制文件也会影响验证的独立性，所以不同采集记录应保存成不同文件。

In [ ]:
train_idx, test_idx, split_method = pipeline.split_rows(groups, y)
source_names = [f["file"] for f in clean_report["files"]]
train_groups = sorted({groups[i] for i in train_idx})
test_groups = sorted({groups[i] for i in test_idx})
print("划分方式:", split_method)
show_table(
    ["部分", "行数", "来源文件", "标签分布"],
    [
        ["训练", len(train_idx), [source_names[i] for i in train_groups],
         dict(Counter(y[train_idx].tolist()))],
        ["验证", len(test_idx), [source_names[i] for i in test_groups],
         dict(Counter(y[test_idx].tolist()))],
    ],
)
assert not set(train_idx).intersection(test_idx)
if split_method == "by_file":
    assert set(train_groups).isdisjoint(test_groups)
fig, ax = plt.subplots(figsize=(11, 2))
ax.scatter(train_idx, np.zeros(len(train_idx)), s=8, color="#5265b0", label="Train")
ax.scatter(test_idx, np.ones(len(test_idx)), s=8, color="#c57e44", label="Validation")
ax.set(xlabel="Row index after cleaning", yticks=[0, 1], yticklabels=["Train", "Validation"],
       title="Separate data before selecting features")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 5. 选择 K 个特征：原始 200 列变成 K 列

这段展开正式脚本 <code>train()</code> 中的特征选择代码：

1. 只允许实际采集扇区的列，或顶部指定的候选列。
2. 排除训练部分中没有变化的常量列。
3. 用训练标签计算每列的 ANOVA F 分数，选分数较高的 K 列。

F 分数比较不同类别之间与类别内部的变化，是单个特征的分类评分。高分不保证选出的组合最优，相关角度可能提供重复信息。官方说明：[f_classif](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html)、[SelectKBest](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html)。

选中集合按原始列顺序排列，这也是模型输入向量必须采用的顺序。

In [ ]:
candidate_indices = (
    pipeline.active_indices()
    if pipeline.FEATURE_CANDIDATE_INDICES is None
    else sorted(set(pipeline.FEATURE_CANDIDATE_INDICES))
)
candidate_indices = np.asarray(
    [i for i in candidate_indices if np.ptp(X[train_idx, i]) > 0], dtype=int
)
if len(candidate_indices) < NOTEBOOK_K:
    raise ValueError(f"只有 {len(candidate_indices)} 个非常量候选列，不能选 K={NOTEBOOK_K}。")

selector = SelectKBest(score_func=f_classif, k=NOTEBOOK_K)
with np.errstate(divide="ignore", invalid="ignore"):
    selector.fit(X[train_idx][:, candidate_indices], y[train_idx])
selected_indices = candidate_indices[selector.get_support(indices=True)]
selected_set = set(selected_indices.tolist())

print("原始列数:", X.shape[1], "；非常量候选:", len(candidate_indices), "；选中:", len(selected_indices))
print("选中特征的原始索引:", selected_indices.tolist())
show_table(
    ["模型输入位置", "原始距离列", "固件读取方式"],
    [(i, int(column), f"x[{i}] = distances[{column}]") for i, column in enumerate(selected_indices)],
)
ranking = np.argsort(np.nan_to_num(selector.scores_, nan=-np.inf))[::-1]
top = ranking[:min(20, len(ranking))]
top_scores = selector.scores_[top]
finite = top_scores[np.isfinite(top_scores)]
cap = max(1.0, float(finite.max()) if len(finite) else 1.0) * 1.1
display_scores = np.nan_to_num(top_scores, nan=0.0, posinf=cap, neginf=0.0)
colors = ["#5265b0" if int(candidate_indices[i]) in selected_set else "#c8cedb" for i in top]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(np.arange(len(top)), display_scores, color=colors)
ax.set_xticks(np.arange(len(top)), candidate_indices[top].astype(str), rotation=45)
ax.set(xlabel="Original feature index", ylabel="ANOVA F score",
       title="Top features; selected columns shown in blue")
for i, score in enumerate(top_scores):
    if np.isposinf(score):
        ax.text(i, display_scores[i], "inf", ha="center", va="bottom")
fig.tight_layout()
plt.show()

## 6. 过滤选中特征缺失的行，构造 float32 输入

未采集扇区的零已经通过候选列限制排除。现在只检查选中的 K 列：默认要求每一个选中距离都大于零，与固件需要有效距离才能推理的约定对应。

这一步不会重新抽取验证集，也不重新选择特征。训练部分仍必须覆盖全部类别。转换为 float32，和固件 float 输入以及 sklearn 的树输入类型对应。

In [ ]:
valid_mask = (
    np.all(X[:, selected_indices] > 0, axis=1)
    if pipeline.REQUIRE_SELECTED_FEATURES_POSITIVE
    else np.ones(len(y), dtype=bool)
)
train_idx = train_idx[valid_mask[train_idx]]
test_idx = test_idx[valid_mask[test_idx]]
if len(test_idx) == 0 or set(y) != set(y[train_idx]) or len(set(y[train_idx])) < 2:
    raise ValueError("选中特征过滤后，验证数据不足或训练类别缺失。")

X_selected = X[:, selected_indices].astype(np.float32)
X_train, y_train = X_selected[train_idx], y[train_idx]
X_test, y_test = X_selected[test_idx], y[test_idx]
print("选中特征缺失而排除的行:", int((~valid_mask).sum()))
show_table(
    ["数组", "形状", "类型"],
    [
        ["X_train", X_train.shape, X_train.dtype],
        ["y_train", y_train.shape, y_train.dtype],
        ["X_test", X_test.shape, X_test.dtype],
        ["y_test", y_test.shape, y_test.dtype],
    ],
)
show_table(
    [*(f"d{int(i)}" for i in selected_indices), "command"],
    [list(map(float, X_train[i])) + [int(y_train[i])] for i in range(min(3, len(X_train)))],
)

## 7. 训练随机森林，再检查验证集

<code>make_classifier()</code> 使用脚本顶部的树数量、深度、叶节点样本数和随机种子。训练阶段只看到 X_train 和 y_train。

预测准确率衡量指令是否完全相同；容差准确率衡量转向指令误差是否在 ±1、±2 等配置范围内。混淆矩阵每行是真实类别，每列是预测类别。模拟数据成绩只用于说明流程。

In [ ]:
validation_classifier = pipeline.make_classifier()
validation_classifier.fit(X_train, y_train)
y_pred = validation_classifier.predict(X_test)
validation_accuracy = float(accuracy_score(y_test, y_pred))
report_text = classification_report(y_test, y_pred, labels=labels.tolist(), zero_division=0)
print(f"验证准确率: {validation_accuracy:.4f}")
print(report_text)
steering = (y_test >= 0) & (y_test <= 15)
if steering.any():
    errors = np.abs(y_pred[steering] - y_test[steering])
    predicted_steering = (y_pred[steering] >= 0) & (y_pred[steering] <= 15)
    for tolerance in pipeline.TOLERANCE_LEVELS:
        score = np.mean(predicted_steering & (errors <= tolerance))
        print(f"转向误差 <= {tolerance}: {score:.4f}")

matrix = confusion_matrix(y_test, y_pred, labels=labels.tolist())
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, cmap="Blues")
ax.set(xticks=np.arange(len(labels)), yticks=np.arange(len(labels)),
       xticklabels=labels, yticklabels=labels,
       xlabel="Predicted command", ylabel="True command", title="Validation confusion matrix")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, int(matrix[i, j]), ha="center", va="center",
                color="white" if matrix[i, j] > matrix.max() / 2 else "#182236")
fig.colorbar(image, ax=ax, label="Rows")
fig.tight_layout()
plt.show()

show_table(
    ["真实指令", "预测指令", "绝对误差"],
    [(int(t), int(p), int(abs(p - t))) for t, p in zip(y_test[:12], y_pred[:12])],
)

## 8. 冻结特征，在全部有效数据上重训最终模型

验证完成后，特征索引不变；用所有选中特征有效的行重训森林，得到要导出的模型。上一步的验证成绩属于验证阶段的森林，不能说成“最终模型在未见数据上的成绩”。

下面同时调用正式 <code>train()</code> 核对一次，确保 Notebook 展开的选择、过滤、验证和重训步骤与脚本结果一致。

In [ ]:
final_samples = X_selected[valid_mask]
final_classifier = pipeline.make_classifier()
final_classifier.fit(final_samples, y[valid_mask])
print("最终森林训练输入:", final_samples.shape)
print("真实类别顺序 classes_:", final_classifier.classes_.tolist())

reference_classifier, reference_indices, reference_samples, reference_scores, reference_split, reference_metrics = (
    pipeline.train(rows, groups, NOTEBOOK_K)
)
assert selected_indices.tolist() == reference_indices
np.testing.assert_array_equal(final_samples, reference_samples)
np.testing.assert_allclose(
    final_classifier.predict_proba(final_samples),
    reference_classifier.predict_proba(final_samples),
    rtol=0, atol=0,
)
assert validation_accuracy == reference_metrics["accuracy"]
print("已核对：逐步示例与正式 train() 的特征、输入、最终概率和验证准确率一致。")

## 9. 查看树如何判断，理解类别编号与真实指令

树通过类似 <code>x[2] <= 1700.5</code> 的判断到达叶节点。这里 x 的下标是选中的 K 维输入位置，要通过 selected_indices 找回原始雷达列。

叶节点保存每类概率。森林平均各棵树的类别概率，然后选最大概率对应的类别。Python 的 predict() 返回真实标签；导出的 C++ predict() 返回类别索引，再由 ModelConfig 转回真实指令。例如训练类别为 {3, 8, 10} 时，索引 {0, 1, 2} 必须映射回 {3, 8, 10}。

In [ ]:
class_commands = final_classifier.classes_.astype(int).tolist()
show_table(
    ["C++ 类别索引", "真实操作指令"],
    list(enumerate(class_commands)),
)
one_sample = final_samples[0:1]
probabilities = final_classifier.predict_proba(one_sample)[0]
class_index = int(np.argmax(probabilities))
print("一行输入:", one_sample[0].tolist())
print("森林平均概率:", probabilities.tolist())
print("C++ 类别索引:", class_index, "-> 真实指令:", class_commands[class_index])
assert class_commands[class_index] == int(final_classifier.predict(one_sample)[0])

print("\n第一棵树的判断：")
print(export_text(
    final_classifier.estimators_[0],
    feature_names=[f"d{int(i)}" for i in selected_indices],
    decimals=1, max_depth=3,
))
print("上面的树中 class 是森林内部类别编码；真实指令看 classes_ 映射。")

## 10. 导出模型头文件，预览实际内容

调用正式脚本 <code>export_header()</code>。同一头文件包含森林判断代码、雷达分辨率、K、原始特征索引、类别映射和采集扇区，供 main.ino 使用。

下面只是创建字符串并预览，未修改任何项目模型文件。头文件是生成产物，应重新生成，不手动编辑或提交 Git。

In [ ]:
header = pipeline.export_header(final_classifier, selected_indices.tolist())
print("头文件字节数:", len(header.encode("utf-8")))
print("\n配置及接口预览：")
print("\n".join(header.splitlines()[:29]))

tree_start = header.index("// tree #1")
tree_end = header.find("// tree #2", tree_start)
print("\n第一棵树的 C++ 片段：")
print(header[tree_start:tree_end if tree_end != -1 else tree_start + 2500])
header_sha256 = hashlib.sha256(header.encode("utf-8")).hexdigest()
print("SHA-256:", header_sha256)

## 11. 用真实 C++ 推理核对 Python

正式脚本 <code>verify_cpp()</code> 用 g++ 编译生成的模型，并在样本、随机输入和树阈值边界上核对真实指令及完整概率。

这一步只在临时目录里写头文件和可执行文件。没有编译器时可以显式设置 RUN_CPP_CHECK=False，此时检查状态会显示 disabled，不能当作验证通过。

In [ ]:
cpp_check = {"status": "disabled"}
if RUN_CPP_CHECK:
    verification_dir = WORKSPACE / "cpp_check"
    verification_dir.mkdir(exist_ok=True)
    cpp_check = pipeline.verify_cpp(
        header, final_classifier, final_samples, verification_dir
    )
show_table(["检查", "结果"], [("C++ 对照", cpp_check)])

## 12. 保存学习输出，核对正式模型未被修改

模拟数据和真实数据模式都保存到本次内核的临时工作目录：头文件、完整清理数据、特征评分、报告和 Python 模型。内核关闭后这些演示文件可能被清理。

正式脚本对应的是“临时生成 → 核对 → publish() 备份和原子替换”，发布阶段失败会恢复已替换的文件。本 Notebook 展示到生成与验证阶段；正式发布使用最后说明中的命令。

In [ ]:
header_path = NOTEBOOK_OUTPUT_DIR / pipeline.HEADER_FILENAME
header_path.write_text(header, encoding="utf-8")
pipeline.write_json(NOTEBOOK_OUTPUT_DIR / pipeline.CLEAN_REPORT_FILENAME, clean_report)
with (NOTEBOOK_OUTPUT_DIR / pipeline.CLEANED_FILENAME).open(
    "w", encoding="utf-8", newline=""
) as handle:
    csv.writer(handle, delimiter=pipeline.INPUT_DELIMITER).writerows(rows)
with (NOTEBOOK_OUTPUT_DIR / pipeline.FEATURE_SCORES_FILENAME).open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.writer(handle)
    writer.writerow(["original_index", "f_score", "selected"])
    writer.writerows(reference_scores)
pipeline.write_json(
    NOTEBOOK_OUTPUT_DIR / pipeline.TRAIN_REPORT_FILENAME,
    {
        "mode": "synthetic_demo" if USE_DEMO_DATA else "real_data_learning",
        "input_dir": str(INPUT_DIR),
        "selected_indices": selected_indices.tolist(),
        "class_commands": class_commands,
        "validation_split": reference_split,
        "validation_metrics": reference_metrics,
        "refit_rows": len(final_samples),
        "export_checks": {"cpp": cpp_check},
        "header_sha256": header_sha256,
        "note": "学习输出，未同步正式模型；模拟数据成绩不代表真实小车效果。",
    },
)
joblib.dump(
    {
        "classifier": final_classifier,
        "selected_indices": selected_indices.tolist(),
        "lidar_point_count": pipeline.LIDAR_POINT_COUNT,
        "class_commands": class_commands,
    },
    NOTEBOOK_OUTPUT_DIR / pipeline.PYTHON_MODEL_FILENAME,
)
show_table(
    ["学习输出文件", "字节数"],
    [(p.name, p.stat().st_size) for p in sorted(NOTEBOOK_OUTPUT_DIR.iterdir())],
)
print("生成的头文件:", header_path)
for path, original_hash in original_model_hashes.items():
    assert file_digest(path) == original_hash, f"正式模型发生变化：{path}"
print("已核对：项目中的两处正式模型文件保持不变。")

## 13. 转到真实数据与正式生成流程

理解完成后：

1. 把采集记录放入 dissertation/dataset，确认距离列数与采集程序一致。
2. 修改 train_model.py 顶部的 K、清理规则和森林参数。
3. 如需在本 Notebook 检查真实数据，将 USE_DEMO_DATA=False，然后重启内核、运行全部单元格。
4. 在项目根目录运行正式脚本，生成完整报告、模型并同步固件：

~~~bash
python software/MLcode/train_model.py
~~~

只生成正式输出、暂不覆盖固件：

~~~bash
python software/MLcode/train_model.py --no-sync
~~~

正式脚本在核对成功后调用 publish()，将头文件写到 software/ML_model，并同步到 software/C_code/selfdrive/main/randomForest.h。模型、清理数据和报告由脚本生成，Git 已忽略这些产物。

**实验变量对照**

| Notebook 中的变量 | 含义 | 正式脚本对应位置 |
| --- | --- | --- |
| rows、groups、clean_report | 清理记录与来源 | load_and_clean() |
| X、y | 完整距离矩阵与真实标签 | train() 的数据准备 |
| train_idx、test_idx | 独立训练 / 验证部分 | split_rows() |
| selected_indices | 原始雷达列索引，固定顺序 | train() 中的 SelectKBest |
| X_selected | 按索引排列的 K 维 float32 输入 | train() |
| validation_classifier | 用于验证的森林 | train() 的首次训练 |
| final_classifier | 冻结特征后重训的森林 | train() 的最终训练 |
| header | 完整模型头文件文本 | export_header() |
| cpp_check | Python / C++ 预测核对 | verify_cpp() |
| NOTEBOOK_OUTPUT_DIR | 学习文件目录 | 正式流程使用 OUTPUT_DIR 和 publish() |